# Real EI Sample Generation (Oregon + South Carolina)

This notebook generates **real** MCMC posterior samples with PyEI and writes:
- `oregon_ei_samples.csv`
- `south_carolina_ei_samples.csv`

Upload files with exact names:
- `or_combined_data.csv`
- `sc_combined_data.csv`


In [ ]:
!pip install pyei --quiet


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from pyei.two_by_two import TwoByTwoEI
from google.colab import files

RACIAL_GROUPS = ["white", "black", "asian", "hispanic"]
print("Imports complete.")


In [ ]:
print("Upload exactly: or_combined_data.csv and sc_combined_data.csv")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:
required_cols = [
    "precinct_id", "total", "white", "black", "asian", "hispanic",
    "democratic_votes", "republican_votes", "total_votes",
]

def load_and_validate(filename):
    df = pd.read_csv(filename)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {filename}: {missing}")

    original_len = len(df)
    df = df[(df["total_votes"] > 0) & (df["total"] > 0)].copy()
    dropped = original_len - len(df)
    if dropped > 0:
        print(f"{filename}: dropped {dropped} rows with zero votes or zero population")

    df = df.reset_index(drop=True)
    print(f"{filename}: loaded {len(df)} precincts")
    return df

oregon_df = load_and_validate("or_combined_data.csv")
sc_df = load_and_validate("sc_combined_data.csv")


In [ ]:
def build_2x2_inputs(df, group_name):
    total_pop = df["total"].values.astype(float)
    group_pop = df[group_name].values.astype(float)
    total_pop_safe = np.where(total_pop == 0, 1.0, total_pop)
    group_fraction = np.clip(group_pop / total_pop_safe, 1e-6, 1.0 - 1e-6)

    tot_votes = df["total_votes"].values.astype(int)
    dem_votes = df["democratic_votes"].values.astype(int)
    tot_safe = np.where(tot_votes == 0, 1, tot_votes).astype(float)
    dem_fraction = np.clip(dem_votes / tot_safe, 1e-6, 1.0 - 1e-6)

    precinct_pops = tot_votes.copy()
    return group_fraction, dem_fraction, precinct_pops


def run_2x2_ei(group_fraction, dem_fraction, precinct_pops, group_name, n_samples=200, n_tune=100):
    print(f"  Running EI for {group_name} (draws={n_samples}, tune={n_tune})")
    ei = TwoByTwoEI(model_name="truncated_normal")
    ei.fit(
        group_fraction=group_fraction,
        votes_fraction=dem_fraction,
        precinct_pops=precinct_pops,
        demographic_group_name=group_name,
        candidate_name="democratic",
        draws=n_samples,
        tune=n_tune,
        target_accept=0.99,
    )
    return ei


def extract_2x2_precinct_estimates(ei, group_fraction, dem_fraction):
    samples = np.array(ei.sampled_voting_prefs)
    b1_mean = float(samples[0, :].mean())
    b2_mean = float(samples[1, :].mean())

    X = group_fraction
    T = dem_fraction
    with np.errstate(divide="ignore", invalid="ignore"):
        b1_precinct = (T - (1.0 - X) * b2_mean) / X

    b1_precinct = np.where(X < 0.02, b1_mean, b1_precinct)
    b1_precinct = np.clip(b1_precinct, 0.0, 1.0)
    return b1_precinct, samples


def compute_2x2_statewide(samples, precinct_pops, group_fraction):
    b1_samples = samples[0, :]

    total_group_pop = float((group_fraction * precinct_pops).sum())
    statewide_dem = b1_samples * total_group_pop
    statewide_rep = (1.0 - b1_samples) * total_group_pop

    p = float((statewide_dem > statewide_rep).mean())
    if p >= 0.5:
        party_of_choice = "democratic"
    else:
        party_of_choice = "republican"
        p = 1.0 - p

    confidence = 1.0 / (1.0 + np.exp(18.0 - 26.0 * p))

    return {
        "party_of_choice": party_of_choice,
        "confidence": round(float(confidence), 4),
        "mean_dem_fraction": round(float(b1_samples.mean()), 6),
        "mean_rep_fraction": round(float(1.0 - b1_samples.mean()), 6),
        "mean_dem_votes": round(float(statewide_dem.mean()), 2),
        "mean_rep_votes": round(float(statewide_rep.mean()), 2),
    }


In [ ]:
def run_for_state(df, state_name, slug, n_samples=200, n_tune=100):
    precinct_out = pd.DataFrame()
    precinct_out["precinct_id"] = df["precinct_id"].values
    precinct_out["state"] = state_name
    precinct_out["total_votes"] = df["total_votes"].values

    statewide_rows = []
    samples_rows = []

    print(f"\n=== {state_name} ===")
    for group_name in RACIAL_GROUPS:
        group_fraction, dem_fraction, precinct_pops = build_2x2_inputs(df, group_name)
        ei = run_2x2_ei(group_fraction, dem_fraction, precinct_pops, group_name, n_samples=n_samples, n_tune=n_tune)
        b1_precinct, samples = extract_2x2_precinct_estimates(ei, group_fraction, dem_fraction)

        precinct_out[f"ei_{group_name}_dem_vote_fraction"] = b1_precinct.round(6)
        precinct_out[f"ei_{group_name}_rep_vote_fraction"] = (1.0 - b1_precinct).round(6)
        precinct_out[f"ei_{group_name}_party_of_choice"] = np.where(b1_precinct >= 0.5, "democratic", "republican")
        precinct_out[f"ei_{group_name}_precinct_confidence"] = np.where(b1_precinct >= 0.5, b1_precinct, 1.0 - b1_precinct).round(4)

        samples_rows.append({
            "state": state_name,
            "racial_group": group_name,
            "b1_samples": ",".join(f"{v:.6f}" for v in samples[0, :].tolist()),
            "b2_samples": ",".join(f"{v:.6f}" for v in samples[1, :].tolist()),
        })

        statewide = compute_2x2_statewide(samples, precinct_pops, group_fraction)
        statewide_rows.append({"state": state_name, "racial_group": group_name, **statewide})

    precinct_file = f"{slug}_ei_precinct.csv"
    statewide_file = f"{slug}_ei_statewide.csv"
    samples_file = f"{slug}_ei_samples.csv"

    precinct_out.to_csv(precinct_file, index=False)
    pd.DataFrame(statewide_rows).to_csv(statewide_file, index=False)
    pd.DataFrame(samples_rows).to_csv(samples_file, index=False)

    print(f"Saved {precinct_file}, {statewide_file}, {samples_file}")
    return precinct_file, statewide_file, samples_file


In [ ]:
# Deadline-safe real settings
N_SAMPLES = 200
N_TUNE = 100

or_files = run_for_state(oregon_df, "Oregon", "oregon", n_samples=N_SAMPLES, n_tune=N_TUNE)
sc_files = run_for_state(sc_df, "South Carolina", "south_carolina", n_samples=N_SAMPLES, n_tune=N_TUNE)

print("\nDone. Output files:")
print(*or_files, *sc_files, sep="\n")


In [ ]:
# Download only the files you need for local JSON regeneration
files.download("oregon_ei_samples.csv")
files.download("south_carolina_ei_samples.csv")
